# 🤖 Model Building

Two ready-to-run models against the silver/gold data:

1. **Trip Duration Regression** — predict trip duration from time features
2. **Station Demand Clustering** — group stations by usage patterns (K-Means)

Export trained artefacts with `export_df` / `export_figure` so they appear in the Streamlit Insights page.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")
from utils import (
    load, load_app_ready, describe, feature_matrix,
    plot_hourly, plot_monthly, plot_top_stations, plot_duration_dist, plot_city_comparison,
    export_df, export_figure,
    CITIES, CITY_DISPLAY_MAP,
)
print("Workspace utils loaded ✓")

## 1 — Trip Duration Regression

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CITY = "oslo"   # oslo | bergen | trondheim (or None for all)
YEAR = None     # int or None
MAX_DURATION_MIN = 60   # cap outliers for cleaner model

df = load(CITY, YEAR)
df = df[df['duration_seconds'].between(60, MAX_DURATION_MIN * 60)].copy()
print(f"Trips after filtering: {len(df):,}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X, y = feature_matrix(df, target='duration_seconds')
y_min = y / 60   # convert to minutes

X_train, X_test, y_train, y_test = train_test_split(X, y_min, test_size=0.2, random_state=42)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Features: {list(X.columns)}")

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f} minutes")
print(f"R²  : {r2:.4f}")

coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_}).sort_values('Coefficient')
coef_df

In [ ]:
# Residual plot
residuals = y_test - y_pred
fig_resid, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred, residuals, alpha=0.3, s=5, color='#1f77b4')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set(xlabel='Predicted (min)', ylabel='Residual (min)', title='Residuals vs Fitted')

axes[1].hist(residuals, bins=60, color='#ff7f0e', alpha=0.85, edgecolor='white')
axes[1].set(xlabel='Residual (min)', ylabel='Count', title='Residual Distribution')

plt.suptitle(f'Trip Duration Model — {CITY.title()} | MAE={mae:.2f} min  R²={r2:.3f}')
plt.tight_layout()

In [ ]:
fig_coef, ax = plt.subplots(figsize=(7, 4))
ax.barh(coef_df['Feature'], coef_df['Coefficient'],
        color=['#d62728' if c < 0 else '#2ca02c' for c in coef_df['Coefficient']], alpha=0.85)
ax.axvline(0, color='grey', linewidth=0.8)
ax.set(title='Feature Coefficients (minutes per unit)', xlabel='Coefficient')
plt.tight_layout()

## 2 — Station Demand Clustering (K-Means)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Build station-level feature matrix: morning peak, evening peak, weekday share
df2 = load(CITY, YEAR)
df2['started_at'] = pd.to_datetime(df2['started_at'], errors='coerce')
df2['hour']       = df2['started_at'].dt.hour
df2['is_weekend'] = (df2['started_at'].dt.dayofweek >= 5).astype(int)

station_features = df2.groupby('start_station_name').agg(
    total_trips    = ('started_at', 'count'),
    morning_share  = ('hour', lambda h: ((h >= 7) & (h <= 9)).mean()),
    evening_share  = ('hour', lambda h: ((h >= 16) & (h <= 19)).mean()),
    weekend_share  = ('is_weekend', 'mean'),
).dropna()

# Drop stations with very few trips
station_features = station_features[station_features['total_trips'] >= 20]
print(f"Stations for clustering: {len(station_features)}")
station_features.describe()

In [ ]:
# Elbow method — choose K
scaler = StandardScaler()
X_scaled = scaler.fit_transform(station_features[['morning_share', 'evening_share', 'weekend_share', 'total_trips']])

inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig_elbow, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), inertias, marker='o', linewidth=2)
ax.set(xlabel='Number of Clusters (K)', ylabel='Inertia', title='Elbow Method')
plt.tight_layout()

In [ ]:
# ── Set K based on the elbow above ────────────────────────────────────────────
K = 3

km = KMeans(n_clusters=K, random_state=42, n_init=10)
station_features['cluster'] = km.fit_predict(X_scaled)

cluster_summary = station_features.groupby('cluster').agg(
    stations       = ('total_trips', 'count'),
    avg_trips      = ('total_trips', 'mean'),
    morning_share  = ('morning_share', 'mean'),
    evening_share  = ('evening_share', 'mean'),
    weekend_share  = ('weekend_share', 'mean'),
).round(3)

print(cluster_summary)

In [ ]:
fig_clusters, ax = plt.subplots(figsize=(8, 5))
colours = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for c in range(K):
    sub = station_features[station_features['cluster'] == c]
    ax.scatter(sub['morning_share'], sub['evening_share'],
               s=sub['total_trips'].clip(upper=5000) / 20,
               alpha=0.7, color=colours[c], label=f'Cluster {c}')

ax.set(xlabel='Morning Peak Share (7–9h)', ylabel='Evening Peak Share (16–19h)',
       title=f'Station Clusters — {CITY.title()} (K={K}, n={len(station_features)})')
ax.legend()
plt.tight_layout()

In [ ]:
# ── Export results ────────────────────────────────────────────────────────────
tag = f"{CITY}_{YEAR or 'all'}"

export_figure(f"model_residuals_{tag}",    fig_resid)
export_figure(f"model_coef_{tag}",         fig_coef)
export_figure(f"cluster_elbow_{tag}",      fig_elbow)
export_figure(f"cluster_scatter_{tag}",    fig_clusters)
export_df(f"cluster_summary_{tag}",        cluster_summary.reset_index())
export_df(f"model_metrics_{tag}",          pd.DataFrame({'MAE': [mae], 'R2': [r2], 'city': [CITY], 'year': [YEAR]}))

print("Exported to gold/notebook_exports/ ✓")
print("Refresh the Insights page in the Streamlit app to see results.")